### Idea 2: otra libreria

In [ ]:
PERIODO_TO_CUATRI = {
    "1er Bimestre": "1er Cuatrimestre",     ##mappea Cursada de Verano a 1erCuatrimestre  (tienen su periodo en 1erBimestre)
    "2do Bimestre": "1er Cuatrimestre",
    "1er Cuatrimestre": "1er Cuatrimestre",
    "3er Bimestre": "2do Cuatrimestre",
    "4to Bimestre": "2do Cuatrimestre",
    "2do Cuatrimestre": "2do Cuatrimestre",
}

In [ ]:
import pandas as pd

file_path = "bases/para tesis LCC_v3.xlsx"
df_totales = pd.read_excel(file_path)
df_planes_por_alumno = pd.read_excel(file_path, sheet_name="planes por alumno")
df_planes_por_alumno["anio_inscripcion"] = pd.to_datetime(df_planes_por_alumno["fecha_inscripcion"]).dt.year
df_planes_por_alumno["plan"] = df_planes_por_alumno["plan"].astype(str).str.strip()

df_cursadas_por_alumno = pd.read_excel(file_path, sheet_name="cursadas por alumno (2)") #pd.read_excel(file_path, sheet_name="cursadas por alumno")
df_cursadas_por_alumno["periodo"] = df_cursadas_por_alumno["periodo"].map(PERIODO_TO_CUATRI).fillna(df_cursadas_por_alumno["periodo"])

df_lista_de_materias = pd.read_excel(file_path, sheet_name="lista de materias")
df_finales_por_alumno = pd.read_excel(file_path, sheet_name="finales por alumno")
df_estados_de_actas = pd.read_excel(file_path, sheet_name="estados de actas")
df_resultados_posibles = pd.read_excel(file_path, sheet_name="resultados posibles")

dic Materias

In [ ]:
dic_materias = df_lista_de_materias[df_lista_de_materias['codigo_materia'].str.contains('COMP93')].sort_values(by="codigo_materia")[:17].drop_duplicates().reset_index().drop(columns=('index')) #dic_materias =dic_materias['codigo_materia'].drop_duplicates().reset_index().drop(columns=('index'))
dic_materias

##### consigo y filtro por una camadaYEAR

In [ ]:
planes_por_alumno = df_planes_por_alumno[df_planes_por_alumno["codigo_carrera"] == "LCCOMP"]

# year_list = [2019]
year_list = range(2013,2025)
primer_anio = (
        planes_por_alumno.groupby("persona")["anio_inscripcion"]
        .min()
        .reset_index()
        .drop_duplicates()
    )

camada = primer_anio[primer_anio["anio_inscripcion"].isin(year_list)]       # año de inscripcion a carreraSelected está en yearList
camadaYEAR = camada[["persona"]]

In [ ]:

# cursaronYEAR = df_cursadas_por_alumno[(df_cursadas_por_alumno['anio_academico'] == year) 
#                                     #  & (df_cursadas_por_alumno['periodo'].isin(['1er Cuatrimestre', '1er Bimestre'])) 
#                                     #  & (df_cursadas_por_alumno['codigo_materia'].isin(['COMP930001', 'COMP930007']))
#                                     #  & (df_cursadas_por_alumno['persona'].isin(df_planes_por_alumno['persona']))  
#                                      ][['persona']].drop_duplicates('persona').copy() #personaID que cursaron en YEAR
# # print(len(cursaron2022))  

# preYEAR = df_cursadas_por_alumno[df_cursadas_por_alumno['anio_academico'] < year][['persona']].drop_duplicates('persona') #personasID con inscripcion  pre-YEAR
# camadaYEAR = cursaronYEAR.drop( cursaronYEAR[cursaronYEAR['persona'].isin( list(preYEAR.transpose().iloc[0].values) )].index ) #camada2022 = cursaron2022 SIN los que se han inscripto pre-YEAR
# camadaYEAR




cursadas_camadaYEAR = df_cursadas_por_alumno[df_cursadas_por_alumno['persona'].isin(list(camadaYEAR.transpose().iloc[0].values)) & 
                                           (df_cursadas_por_alumno['codigo_materia'].isin(dic_materias['codigo_materia'].tolist())) & 
                                           (df_cursadas_por_alumno['resultado'] != 'U')]
cursadas_camadaYEAR

cursadas_camadaYEAR.insert(len(cursadas_camadaYEAR.columns), 'vector', [[0]*len(dic_materias) for i in range(len(cursadas_camadaYEAR)) ]  )
cursadas_camadaYEAR = cursadas_camadaYEAR.sort_values(['persona', 'anio_academico', 'periodo' ] ) # hago sort 
cursadas_camadaYEAR
cursadas_camadaYEAR

In [ ]:
len(cursadas_camadaYEAR[['persona']].drop_duplicates())*0.07

In [ ]:
primer_anio.groupby(["anio_inscripcion"]).agg(gente=("persona", "nunique"))

In [ ]:
for filaNro in range(1, len(cursadas_camadaYEAR)):
    oldFila = cursadas_camadaYEAR.iloc[filaNro-1]
    fila = cursadas_camadaYEAR.iloc[filaNro]

    if int(oldFila.persona) == int(fila.persona):
        cursadas_camadaYEAR.at[int(fila.name), 'vector']  = oldFila.vector

    # if fila.resultado == 'A':
    newVect = cursadas_camadaYEAR.loc[int(fila.name), 'vector'].copy()
    k = int(dic_materias[dic_materias['codigo_materia'] == fila.codigo_materia ]['codigo_materia'].index[0])
    newVect[k] = 1
    cursadas_camadaYEAR.at[int(fila.name), 'vector'] = newVect


cursadas_camadaYEAR

In [ ]:
cursadas_camadaYEAR['vector_str'] = cursadas_camadaYEAR['vector'].apply(lambda x: "".join(map(str, x)))
cursadas_camadaYEAR

In [ ]:
#cursadas_camadaYEAR[['persona','periodo','anio_academico']].drop_duplicates()

In [ ]:
# test year mayor o igual a rango
# year = 2015
# cursaronYEAR = df_cursadas_por_alumno[(df_cursadas_por_alumno['anio_academico'] >= year) 
#                                      #& (df_cursadas_por_alumno['periodo'].isin(['1er Cuatrimestre', '1er Bimestre'])) 
#                                      & (df_cursadas_por_alumno['codigo_materia'].isin(['COMP930001', 'COMP930007']))
#                                      & (df_cursadas_por_alumno['persona'].isin(df_planes_por_alumno['persona']))  ][['persona']].drop_duplicates('persona').copy() #personaID que cursaron en YEAR
# # print(len(cursaron2022))  

# preYEAR = df_cursadas_por_alumno[df_cursadas_por_alumno['anio_academico'] < year][['persona']].drop_duplicates('persona') #personasID con inscripcion  pre-YEAR
# camadaYEAR = cursaronYEAR.drop( cursaronYEAR[cursaronYEAR['persona'].isin( list(preYEAR.transpose().iloc[0].values) )].index ) #camada2022 = cursaron2022 SIN los que se han inscripto pre-YEAR
# camadaYEAR

# cursada_camadaYEAR = df_cursadas_por_alumno[(df_cursadas_por_alumno['resultado'] == 'A') & 
#                                             (df_cursadas_por_alumno['persona'].isin(camadaYEAR['persona'].to_list())) & 
#                                             (df_cursadas_por_alumno['codigo_materia'].isin(dic_materias['codigo_materia'].tolist()) )]
# cursada_camadaYEAR

hay 100 estudiantes / trayectorias de materias (hay 126 en la camada, pero se filtro por aprobados)

In [ ]:
trayectorias_posibles =  len(cursadas_camadaYEAR['persona'].drop_duplicates() )
trayectorias_posibles

sorting por persona, year, periodo

In [ ]:
#estudiantesTest = [7028,5290,5355,2703,731, 25964,6202, 23993, 2529, 14352]
df_test_cursadas = cursadas_camadaYEAR.sort_values(['persona', 'anio_academico', 'periodo' ] ) # & (df_cursadas_por_alumno['persona'].isin(estudiantesTest))  ]
print(len(df_test_cursadas))
df_test_cursadas.head()

In [ ]:
#OldVer: antes solo por materia
#student_careers = df_test_cursadas.groupby('persona')['codigo_materia'].apply(list)

#NewVer: ahora por snapshoot de cuatri
student_careers = df_test_cursadas.groupby('persona')['vector_str'].apply(list)

# 4. Convert the Pandas Series into a native Python "list of lists"
# PrefixSpan needs pure Python lists, not Pandas objects
sequence_database = student_careers.tolist()
sequence_database

##### nuevo hago restriccion para que solo quede el vector agrupado por ["persona", "anio_academico", "periodo"]

In [ ]:
df_test_cursadas = (
    df_test_cursadas.groupby(["persona", "anio_academico", "periodo"], sort=False)["vector_str"]
    .last()   # o .max(), si querés quedarte con el estado más "avanzado" de ese cuatri
    .reset_index()
)
orden_periodo = {
    "1er Cuatrimestre": 1,
    "2do Cuatrimestre": 2,
}
df_test_cursadas["periodo"] = df_test_cursadas["periodo"].map(orden_periodo)
df_test_cursadas = df_test_cursadas.sort_values(["persona", "anio_academico", "periodo"])

# 4. Secuencia final por persona: lista de vectStr en orden cronológico
sequence_database = (
    df_test_cursadas.groupby("persona")["vector_str"]
    .apply(list)
    .tolist()
)

In [ ]:
from prefixspan import PrefixSpan

# Sample dataset: a list of sequences
# db = [
#     [0, 1, 2, 3, 4],
#     [1, 1, 1, 3, 4],
#     [2, 1, 2, 2, 0],
#     [1, 1, 1, 2, 2],
# ]

ps = PrefixSpan(sequence_database)

# 1. Find subsequences that appear at least 2 times
#print(ps.frequent(5) )

# 2. Find the top 5 most frequent subsequences
#print(ps.topk(5) )


In [ ]:
trayectorias_posibles

In [ ]:
materiasCatalogo = pd.merge(dic_materias, df_lista_de_materias, on='codigo_materia', how='left')
materiasCatalogo['codigo'] = ['alg','algo1','algo2','algo3','analisis', 'bd', 'inge1','inge2', 'LyC','metNum','orga1','orga2','plp','proba','SO','redes', 'tleng']

def strMateriasAprobadas(l): #l <- nodosCaminos.iloc[i]['vector']
    boolean_mask = [bool(int(valor)) for valor in l]
    # temp= pd.merge(dic_materias[boolean_mask], df_lista_de_materias, on='codigo_materia', how='left')
    output = f""
    for i in range(len(materiasCatalogo[boolean_mask])): #  materiasCatalogo[boolean_mask]['codigo'].tolist():
        output+= str(materiasCatalogo[boolean_mask].iloc[i].name+1) +'. '+ materiasCatalogo[boolean_mask].iloc[i].codigo +', '
    return output[:len(output)-1]

# strMateriasAprobadas(['1', '0', '0', '0', '1', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0'])

In [ ]:
def parseSnapshot( listSnaps ):

    out = []
    for snp in listSnaps:
        lst = list(snp)        

        out.append( "{" + strMateriasAprobadas(lst)[:-1] + "}")

        # for i in range(len(lst)): 
        #     if lst[i] == "1" :
        #         output += dic_materias.iloc[i].nombre_materia + ","
        # out.append( output[:-1]+"}")
    return out

In [ ]:
len(sequence_database)

In [ ]:
arraysTotales = 0
for i in sequence_database:
    arraysTotales +=len(i)
arraysTotales

In [ ]:
trayectorias_posibles*0.10

In [ ]:
patterns = ps.frequent(trayectorias_posibles*0.10)

# Sort by the length of the pattern list (x[1]), in descending order
longest_first = sorted(patterns, key=lambda x: len(x[1]), reverse=True)

for support, pattern in longest_first[:20]:
    print(f"Length: {len(pattern)} | Support: {support} | Pattern: {' -> '.join( pattern )}")
    print(f"Length: {len(pattern)} | Support: {support} | Pattern: {' -> '.join( parseSnapshot(pattern) )} ")
    print("")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

x = [len(item[1]) for item in patterns]
y = [item[0] for item in patterns]

# size and color:
#colors = np.random.uniform(15, 80, len(x))


plt.scatter(x, y, s=10, vmin=0, vmax=100)
plt.xlabel("Sequence Pattern Lenght")
plt.ylabel("Support / frecuencia")

In [ ]:
df_patterns = pd.DataFrame({ "Lenght":  x, "Support": y, "Pattern": [item[1] for item in patterns] })
df_patterns.sort_values(['Lenght', 'Support' ], ascending=False )

### VectStr para plan estudio

In [ ]:
dic_materias.sort_values("codigo_materia")

dic_materia_idx={codigo: idx for idx, codigo in enumerate(dic_materias["codigo_materia"])}
dic_materia_idx
m = len(dic_materia_idx)

In [ ]:
df = pd.read_csv("C:/Users/Leila/Documents/uba/tesis/git/tesis/repo/visualization_prod/plan/LCCOMP_1993.csv") 

In [ ]:
df["persona"] = [10 for _ in range(len(df))]
df["is_approved"] = [1 for _ in range(len(df))]
df["materia_pos"] = df["codigo_materia"].map(dic_materia_idx)

In [ ]:
wide = (
    df
    .pivot_table(
        index=["persona", "cuatrimestre_sugerido"],
        columns="materia_pos",
        values="is_approved",
        aggfunc="max",
        fill_value=0,
    )
    .reindex(columns=range(m), fill_value=0)
    .astype(np.int8)
)
wide.index.set_names(["persona", "step_nro"], inplace=True)

In [ ]:
wide

In [ ]:
def vectorEstado_to_vectStr(vector) -> str:
    """Representación en vectStr y hasheable de un vectorEstado """ #dict O(1)
    return np.asarray(vector, dtype=np.int8).tobytes().hex()

In [ ]:
# full_index = pd.MultiIndex.from_frame(
#     step_order[[persona_column, "step_nro"]].drop_duplicates(),
#     names=[persona_column, "step_nro"],
# )
# wide_full = wide.reindex(full_index, fill_value=0).astype(np.int8)
# wide_full = wide_full.sort_index(level=[persona_column, "step_nro"])

state_matrix = wide.groupby(level="persona").cummax()
state_matrix["vectStr"] = [vectorEstado_to_vectStr(row) for row in state_matrix.to_numpy(dtype=np.int8)]
state_matrix = state_matrix.reset_index() # ACA EMPIEZA EDIT.sort_values([persona_column, "step_nro"])


In [ ]:
state_matrix